# MLP — Brain Tumor MRI Classification

**Model Owner:** Thennakoon T.M.A.N  
**IT Number:** IT25100854  
**Assigned Model:** MLP

This notebook contains **two separately trained variants** of the assigned model.  
The better variant is selected using **Validation Macro F1**, and the Official Test set is used only after variant selection.

## 1. Project and Model Overview

The task is four-class MRI classification: glioma, meningioma, no-tumor and pituitary.

The MLP does not learn spatial convolutional features directly from the full image. Instead, it receives engineered **HOG + GLCM** features that are standardized and reduced using PCA.

## 2. Why MLP Is Suitable + Final Evaluation Coverage

**Why MLP?**  
An MLP is a fully connected neural network that can learn nonlinear relationships from engineered image features. It provides a useful comparison against CNN architectures.

This notebook includes:
- model suitability,
- preprocessing / feature engineering,
- two manually tuned variants,
- separate training and visualization,
- Accuracy / Loss / Macro F1,
- Validation-based selection,
- Official Test evaluation,
- Classification Report and Confusion Matrix,
- limitations and improvements,
- `.keras`, CSV, scaler, PCA and metadata exports.

## 3. Install Required Libraries

In [ ]:
%pip -q install kagglehub scikit-image scikit-learn joblib

## 4. Import Libraries and Set Reproducibility

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile
import random
import time
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from skimage.color import rgb2gray
from skimage.feature import hog, graycomatrix, graycoprops
from skimage.transform import resize

import joblib
import kagglehub

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

KAGGLE_DATASET = "maleeshaweerakkody/brain-tumor-mri-preprocessed"

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
# ------------------------------------------------------------
# Model ownership
# ------------------------------------------------------------
MODEL_OWNER = "Thennakoon T.M.A.N"
OWNER_IT_NUMBER = "IT25100854"
OWNER_MEMBER_NO = OWNER_MEMBER_NO

print("Model Owner:", MODEL_OWNER)
print("IT Number:", OWNER_IT_NUMBER)
print("Assigned Model:", "MLP")


## 5. Download the Final Preprocessed Kaggle Dataset

In [ ]:
KAGGLE_ROOT = Path(
    kagglehub.dataset_download(
        KAGGLE_DATASET
    )
)

print("Downloaded dataset:")
print(KAGGLE_ROOT)

WORK = Path(
    "/content/brain_tumor_mlp_svm"
)

DATA_ROOT = (
    WORK
    / "dataset"
)

MODELS_DIR = (
    WORK
    / "models"
)

RESULTS_DIR = (
    WORK
    / "results"
)

for p in [
    WORK,
    MODELS_DIR,
    RESULTS_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

if DATA_ROOT.exists():
    shutil.rmtree(
        DATA_ROOT
    )

DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

zip_files = list(
    KAGGLE_ROOT.rglob(
        "*.zip"
    )
)

if zip_files:
    for zip_path in zip_files:
        print(
            "Extracting:",
            zip_path
        )

        with zipfile.ZipFile(
            zip_path,
            "r"
        ) as z:
            z.extractall(
                DATA_ROOT
            )

    SEARCH_ROOT = DATA_ROOT

else:
    SEARCH_ROOT = KAGGLE_ROOT

print(
    "Search root:",
    SEARCH_ROOT
)

## 6. Detect Training, Validation and Official Test Folders

In [ ]:
def find_split_dir(
    root,
    possible_names
):
    names = {
        x.lower()
        for x in possible_names
    }

    matches = [
        p
        for p in root.rglob("*")
        if p.is_dir()
        and p.name.lower() in names
    ]

    if not matches:
        return None

    return sorted(
        matches,
        key=lambda p: (
            -len(
                [
                    x
                    for x in p.iterdir()
                    if x.is_dir()
                ]
            ),
            len(p.parts)
        )
    )[0]


TRAIN_DIR = find_split_dir(
    SEARCH_ROOT,
    ["train", "training"]
)

VAL_DIR = find_split_dir(
    SEARCH_ROOT,
    ["validation", "valid", "val"]
)

TEST_DIR = find_split_dir(
    SEARCH_ROOT,
    ["test", "testing"]
)

print("Training  :", TRAIN_DIR)
print("Validation:", VAL_DIR)
print("Testing   :", TEST_DIR)

assert TRAIN_DIR is not None
assert VAL_DIR is not None
assert TEST_DIR is not None

## 7. Exploratory Data Analysis and Class Distribution

In [ ]:
IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff"
}


def get_class_names(
    root
):
    return sorted(
        [
            p.name
            for p in root.iterdir()
            if p.is_dir()
        ]
    )


CLASS_NAMES = get_class_names(
    TRAIN_DIR
)

NUM_CLASSES = len(
    CLASS_NAMES
)

print(
    "Class order:",
    CLASS_NAMES
)

assert get_class_names(
    VAL_DIR
) == CLASS_NAMES

assert get_class_names(
    TEST_DIR
) == CLASS_NAMES


def count_images(
    root
):
    rows = []

    for class_name in CLASS_NAMES:
        class_dir = (
            root
            / class_name
        )

        count = sum(
            1
            for p in class_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTS
        )

        rows.append({
            "Class": class_name,
            "Count": count
        })

    return pd.DataFrame(
        rows
    )


train_counts = count_images(
    TRAIN_DIR
).rename(
    columns={
        "Count": "Training"
    }
)

val_counts = count_images(
    VAL_DIR
).rename(
    columns={
        "Count": "Validation"
    }
)

test_counts = count_images(
    TEST_DIR
).rename(
    columns={
        "Count": "Testing"
    }
)

class_counts = (
    train_counts
    .merge(
        val_counts,
        on="Class"
    )
    .merge(
        test_counts,
        on="Class"
    )
)

display(
    class_counts
)

class_counts.set_index(
    "Class"
)[
    [
        "Training",
        "Validation",
        "Testing"
    ]
].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title(
    "Brain Tumor MRI — Class Distribution"
)

plt.ylabel(
    "Number of Images"
)

plt.xticks(
    rotation=20
)

plt.tight_layout()
plt.show()

## 8. MLP-Specific Preprocessing / Feature Engineering

```text
Final group-preprocessed MRI
        ↓
Grayscale + 64×64 feature image
        ↓
HOG features
+
GLCM texture features
        ↓
1769 combined features
        ↓
StandardScaler (fit on Training only)
        ↓
PCA 95% variance (fit on Training only)
        ↓
MLP
```

The final Kaggle images already contain the group's earlier resize/padding and CLAHE preprocessing, so CLAHE is not applied again here.

In [ ]:
FEATURE_IMAGE_SIZE = (
    64,
    64
)

HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (
    8,
    8
)
HOG_CELLS_PER_BLOCK = (
    2,
    2
)

GLCM_DISTANCES = [
    1
]

GLCM_ANGLES = [
    0,
    np.pi / 4,
    np.pi / 2,
    3 * np.pi / 4
]

GLCM_PROPERTIES = [
    "contrast",
    "dissimilarity",
    "homogeneity",
    "energy",
    "correlation"
]

print(
    "Feature image size:",
    FEATURE_IMAGE_SIZE
)

print(
    "HOG orientations:",
    HOG_ORIENTATIONS
)

print(
    "GLCM properties:",
    GLCM_PROPERTIES
)

## 9. Extract HOG + GLCM Features from Train / Validation / Test

In [ ]:
def load_for_features(
    image_path
):
    image = (
        Image.open(
            image_path
        )
        .convert(
            "RGB"
        )
    )

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    # This is a model-feature preparation step only.
    # The final Kaggle image has already received the group's
    # resize/padding + CLAHE preprocessing.
    gray = rgb2gray(
        image
    )

    gray64 = resize(
        gray,
        FEATURE_IMAGE_SIZE,
        anti_aliasing=True
    )

    gray64_uint8 = np.clip(
        gray64 * 255.0,
        0,
        255
    ).astype(
        np.uint8
    )

    return (
        image,
        gray64,
        gray64_uint8
    )


def extract_hog_glcm(
    image_path
):
    _, gray64, gray64_uint8 = (
        load_for_features(
            image_path
        )
    )

    hog_features = hog(
        gray64,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm="L2-Hys",
        feature_vector=True
    )

    glcm = graycomatrix(
        gray64_uint8,
        distances=GLCM_DISTANCES,
        angles=GLCM_ANGLES,
        levels=256,
        symmetric=True,
        normed=True
    )

    # Average each GLCM property across the four angles.
    glcm_features = np.array(
        [
            graycoprops(
                glcm,
                prop
            ).mean()
            for prop in GLCM_PROPERTIES
        ],
        dtype=np.float32
    )

    combined = np.concatenate(
        [
            hog_features.astype(
                np.float32
            ),
            glcm_features
        ]
    )

    return combined


def collect_split(
    root
):
    X = []
    y = []
    paths = []

    for class_index, class_name in enumerate(
        CLASS_NAMES
    ):
        class_dir = (
            root
            / class_name
        )

        image_paths = sorted(
            [
                p
                for p in class_dir.rglob("*")
                if p.is_file()
                and p.suffix.lower()
                in IMAGE_EXTS
            ]
        )

        for image_path in image_paths:
            X.append(
                extract_hog_glcm(
                    image_path
                )
            )

            y.append(
                class_index
            )

            paths.append(
                str(
                    image_path
                )
            )

    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            y,
            dtype=np.int64
        ),
        paths
    )


start = time.time()

X_train_raw, y_train, train_paths = collect_split(
    TRAIN_DIR
)

X_val_raw, y_val, val_paths = collect_split(
    VAL_DIR
)

X_test_raw, y_test, test_paths = collect_split(
    TEST_DIR
)

print(
    "Feature extraction time:",
    round(
        time.time() - start,
        2
    ),
    "seconds"
)

print(
    "Training feature shape:",
    X_train_raw.shape
)

print(
    "Validation feature shape:",
    X_val_raw.shape
)

print(
    "Testing feature shape:",
    X_test_raw.shape
)

print(
    "Combined HOG + GLCM features per image:",
    X_train_raw.shape[
        1
    ]
)

assert (
    X_train_raw.shape[
        1
    ]
    == 1769
), (
    "Expected 1769 HOG+GLCM features. "
    f"Got {X_train_raw.shape[1]}."
)

In [ ]:
sample_path = Path(
    train_paths[
        0
    ]
)

original_rgb, gray64, gray64_uint8 = load_for_features(
    sample_path
)

hog_vector, hog_image = hog(
    gray64,
    orientations=HOG_ORIENTATIONS,
    pixels_per_cell=HOG_PIXELS_PER_CELL,
    cells_per_block=HOG_CELLS_PER_BLOCK,
    block_norm="L2-Hys",
    visualize=True,
    feature_vector=True
)

plt.figure(
    figsize=(6, 5)
)

plt.imshow(
    original_rgb
)

plt.title(
    "Final Group-Preprocessed MRI Input"
)

plt.axis(
    "off"
)

plt.tight_layout()
plt.show()


plt.figure(
    figsize=(6, 5)
)

plt.imshow(
    gray64,
    cmap="gray"
)

plt.title(
    "Model Feature Preparation — Grayscale 64×64"
)

plt.axis(
    "off"
)

plt.tight_layout()
plt.show()


plt.figure(
    figsize=(6, 5)
)

plt.imshow(
    hog_image,
    cmap="gray"
)

plt.title(
    "HOG Feature Visualization"
)

plt.axis(
    "off"
)

plt.tight_layout()
plt.show()


glcm_demo = graycomatrix(
    gray64_uint8,
    distances=GLCM_DISTANCES,
    angles=GLCM_ANGLES,
    levels=256,
    symmetric=True,
    normed=True
)

glcm_demo_values = {
    prop: float(
        graycoprops(
            glcm_demo,
            prop
        ).mean()
    )
    for prop in GLCM_PROPERTIES
}

print(
    "GLCM feature values:"
)

display(
    pd.DataFrame(
        [
            glcm_demo_values
        ]
    )
)

## 10. StandardScaler + PCA — Leakage-Safe Preprocessing

Both `StandardScaler` and PCA are **fit only on Training features**. Validation and Test are transformed using the already-fitted objects.

In [ ]:
# ============================================================
# LEAKAGE-SAFE FEATURE NORMALIZATION + PCA
# Fit ONLY on Training data.
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_raw
)

X_val_scaled = scaler.transform(
    X_val_raw
)

X_test_scaled = scaler.transform(
    X_test_raw
)


pca = PCA(
    n_components=0.95,
    svd_solver="full"
)

X_train = pca.fit_transform(
    X_train_scaled
)

X_val = pca.transform(
    X_val_scaled
)

X_test = pca.transform(
    X_test_scaled
)

print(
    "Original feature count:",
    X_train_raw.shape[
        1
    ]
)

print(
    "PCA feature count:",
    X_train.shape[
        1
    ]
)

print(
    "Explained variance retained:",
    float(
        pca.explained_variance_ratio_.sum()
    )
)


plt.figure(
    figsize=(8, 4)
)

plt.plot(
    np.cumsum(
        pca.explained_variance_ratio_
    )
)

plt.title(
    "PCA Cumulative Explained Variance"
)

plt.xlabel(
    "Number of Principal Components"
)

plt.ylabel(
    "Cumulative Explained Variance"
)

plt.tight_layout()
plt.show()


SCALER_PATH = (
    MODELS_DIR
    / "hog_glcm_standard_scaler.joblib"
)

PCA_PATH = (
    MODELS_DIR
    / "hog_glcm_pca_95.joblib"
)

joblib.dump(
    scaler,
    SCALER_PATH
)

joblib.dump(
    pca,
    PCA_PATH
)

print(
    "Saved scaler:",
    SCALER_PATH
)

print(
    "Saved PCA:",
    PCA_PATH
)

# One-hot labels are used only for Keras MLP training so
# tf.keras.metrics.F1Score can calculate Macro F1 correctly.
# Integer y_train / y_val / y_test are retained for sklearn evaluation.

Y_train_mlp = tf.keras.utils.to_categorical(
    y_train,
    num_classes=NUM_CLASSES
)

Y_val_mlp = tf.keras.utils.to_categorical(
    y_val,
    num_classes=NUM_CLASSES
)

Y_test_mlp = tf.keras.utils.to_categorical(
    y_test,
    num_classes=NUM_CLASSES
)

print("MLP one-hot Training labels:", Y_train_mlp.shape)
print("MLP one-hot Validation labels:", Y_val_mlp.shape)


## 11. Manual Hyperparameter Tuning Plan

**Variant A — Baseline MLP**
- Hidden layers: 128 → 64
- Dropout: 0.30
- Learning rate: 0.001

**Variant B — Regularized Deep MLP**
- Hidden layers: 256 → 128 → 64
- Batch Normalization
- Dropout: 0.40
- Learning rate: 0.0003

The winning variant is selected using **Validation Macro F1**.

### Variant names used in this project
- **Variant A — Baseline MLP**
- **Variant B — Regularized Deep MLP**


## 12. Build the Two MLP Variants

In [ ]:
MODEL_NAME = "MLP"

VARIANT_NAMES = {
    "A": "Baseline MLP",
    "B": "Regularized Deep MLP",
}

VARIANTS = {
    "A": {
        "hidden_units": [
            128,
            64
        ],
        "dropout": 0.30,
        "learning_rate": 1e-3,
        "batch_norm": False,
    },

    "B": {
        "hidden_units": [
            256,
            128,
            64
        ],
        "dropout": 0.40,
        "learning_rate": 3e-4,
        "batch_norm": True,
    },
}


def new_macro_f1():
    return tf.keras.metrics.F1Score(
        average="macro",
        name="f1_macro"
    )


def build_mlp(
    variant_name
):
    config = VARIANTS[
        variant_name
    ]

    inputs = keras.Input(
        shape=(
            X_train.shape[
                1
            ],
        )
    )

    x = inputs

    for units in config[
        "hidden_units"
    ]:
        x = layers.Dense(
            units,
            activation="relu"
        )(x)

        if config[
            "batch_norm"
        ]:
            x = layers.BatchNormalization()(x)

        x = layers.Dropout(
            config[
                "dropout"
            ]
        )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name=(
            f"MLP_"
            f"Variant_"
            f"{variant_name}"
        )
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=config[
                "learning_rate"
            ]
        ),
        loss=keras.losses.CategoricalCrossentropy(),
        metrics=[
            "accuracy",
            new_macro_f1()
        ]
    )

    return (
        model,
        config
    )

## 13. Train Variant A Separately — Baseline MLP

In [ ]:
VARIANT_A_NAME = VARIANT_NAMES[
    "A"
]

print(
    "TRAINING MLP VARIANT A —",
    VARIANT_A_NAME
)

mlp_a, config_a = build_mlp(
    "A"
)

MLP_A_PATH = (
    MODELS_DIR
    / "mlp_variant_A_best.keras"
)

callbacks_a = [
    keras.callbacks.ModelCheckpoint(
        MLP_A_PATH,
        monitor="val_f1_macro",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_f1_macro",
        mode="max",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

history_a = mlp_a.fit(
    X_train,
    Y_train_mlp,
    validation_data=(
        X_val,
        Y_val_mlp
    ),
    epochs=80,
    batch_size=32,
    callbacks=callbacks_a,
    verbose=1
)

history_a_df = pd.DataFrame(
    history_a.history
)

best_mlp_a = (
    keras.models.load_model(
        MLP_A_PATH
    )
)

a_val_probs = (
    best_mlp_a.predict(
        X_val,
        verbose=0
    )
)

a_val_pred = np.argmax(
    a_val_probs,
    axis=1
)

MLP_A_METRICS = {
    "Variant": "A",
    "Variant Name": VARIANT_A_NAME,
    "Validation Accuracy": float(
        accuracy_score(
            y_val,
            a_val_pred
        )
    ),
    "Validation Precision": float(
        precision_score(
            y_val,
            a_val_pred,
            average="macro",
            zero_division=0
        )
    ),
    "Validation Recall": float(
        recall_score(
            y_val,
            a_val_pred,
            average="macro",
            zero_division=0
        )
    ),
    "Validation F1": float(
        f1_score(
            y_val,
            a_val_pred,
            average="macro",
            zero_division=0
        )
    ),
}

display(
    pd.DataFrame([
        MLP_A_METRICS
    ])
)

## 14. Variant A Visualization — Accuracy, Loss and Macro F1

In [ ]:
epochs = np.arange(
    1,
    len(history_a_df) + 1
)

plt.figure(
    figsize=(8, 4)
)

plt.plot(
    epochs,
    history_a_df[
        "accuracy"
    ],
    label="Training Accuracy"
)

plt.plot(
    epochs,
    history_a_df[
        "val_accuracy"
    ],
    label="Validation Accuracy"
)

plt.title(
    "MLP Variant A — Baseline MLP — Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(
    figsize=(8, 4)
)

plt.plot(
    epochs,
    history_a_df[
        "loss"
    ],
    label="Training Loss"
)

plt.plot(
    epochs,
    history_a_df[
        "val_loss"
    ],
    label="Validation Loss"
)

plt.title(
    "MLP Variant A — Baseline MLP — Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(
    figsize=(8, 4)
)

plt.plot(
    epochs,
    history_a_df[
        "f1_macro"
    ],
    label="Training Macro F1"
)

plt.plot(
    epochs,
    history_a_df[
        "val_f1_macro"
    ],
    label="Validation Macro F1"
)

plt.title(
    "MLP Variant A — Baseline MLP — Macro F1"
)

plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Train Variant B Separately — Regularized Deep MLP

In [ ]:
VARIANT_B_NAME = VARIANT_NAMES[
    "B"
]

print(
    "TRAINING MLP VARIANT B —",
    VARIANT_B_NAME
)

mlp_b, config_b = build_mlp(
    "B"
)

MLP_B_PATH = (
    MODELS_DIR
    / "mlp_variant_B_best.keras"
)

callbacks_b = [
    keras.callbacks.ModelCheckpoint(
        MLP_B_PATH,
        monitor="val_f1_macro",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_f1_macro",
        mode="max",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

history_b = mlp_b.fit(
    X_train,
    Y_train_mlp,
    validation_data=(
        X_val,
        Y_val_mlp
    ),
    epochs=80,
    batch_size=32,
    callbacks=callbacks_b,
    verbose=1
)

history_b_df = pd.DataFrame(
    history_b.history
)

best_mlp_b = (
    keras.models.load_model(
        MLP_B_PATH
    )
)

b_val_probs = (
    best_mlp_b.predict(
        X_val,
        verbose=0
    )
)

b_val_pred = np.argmax(
    b_val_probs,
    axis=1
)

MLP_B_METRICS = {
    "Variant": "B",
    "Variant Name": VARIANT_B_NAME,
    "Validation Accuracy": float(
        accuracy_score(
            y_val,
            b_val_pred
        )
    ),
    "Validation Precision": float(
        precision_score(
            y_val,
            b_val_pred,
            average="macro",
            zero_division=0
        )
    ),
    "Validation Recall": float(
        recall_score(
            y_val,
            b_val_pred,
            average="macro",
            zero_division=0
        )
    ),
    "Validation F1": float(
        f1_score(
            y_val,
            b_val_pred,
            average="macro",
            zero_division=0
        )
    ),
}

display(
    pd.DataFrame([
        MLP_B_METRICS
    ])
)

## 16. Variant B Visualization — Accuracy, Loss and Macro F1

In [ ]:
epochs = np.arange(
    1,
    len(history_b_df) + 1
)

plt.figure(
    figsize=(8, 4)
)

plt.plot(
    epochs,
    history_b_df[
        "accuracy"
    ],
    label="Training Accuracy"
)

plt.plot(
    epochs,
    history_b_df[
        "val_accuracy"
    ],
    label="Validation Accuracy"
)

plt.title(
    "MLP Variant B — Regularized Deep MLP — Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(
    figsize=(8, 4)
)

plt.plot(
    epochs,
    history_b_df[
        "loss"
    ],
    label="Training Loss"
)

plt.plot(
    epochs,
    history_b_df[
        "val_loss"
    ],
    label="Validation Loss"
)

plt.title(
    "MLP Variant B — Regularized Deep MLP — Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(
    figsize=(8, 4)
)

plt.plot(
    epochs,
    history_b_df[
        "f1_macro"
    ],
    label="Training Macro F1"
)

plt.plot(
    epochs,
    history_b_df[
        "val_f1_macro"
    ],
    label="Validation Macro F1"
)

plt.title(
    "MLP Variant B — Regularized Deep MLP — Macro F1"
)

plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.legend()
plt.tight_layout()
plt.show()

## 17. Compare Variant A vs Variant B and Select the Best MLP

In [ ]:
variant_results = pd.DataFrame([
    MLP_A_METRICS,
    MLP_B_METRICS
]).sort_values(
    "Validation F1",
    ascending=False
).reset_index(
    drop=True
)

display(
    variant_results
)

for metric_name in [
    "Validation Accuracy",
    "Validation Precision",
    "Validation Recall",
    "Validation F1"
]:
    plt.figure(
        figsize=(7, 4)
    )

    plt.bar(
        variant_results[
            "Variant Name"
        ],
        variant_results[
            metric_name
        ]
    )

    plt.title(
        f"MLP Variant Comparison — {metric_name}"
    )

    plt.ylabel(
        metric_name
    )

    plt.xticks(
        rotation=15
    )

    plt.tight_layout()
    plt.show()


BEST_VARIANT = str(
    variant_results.loc[
        0,
        "Variant"
    ]
)

BEST_VARIANT_NAME = str(
    variant_results.loc[
        0,
        "Variant Name"
    ]
)

if BEST_VARIANT == "A":
    BEST_MLP_CHECKPOINT = MLP_A_PATH
else:
    BEST_MLP_CHECKPOINT = MLP_B_PATH

FINAL_MODEL_PATH = (
    MODELS_DIR
    / "it25100854_mlp_selected.keras"
)

shutil.copy2(
    BEST_MLP_CHECKPOINT,
    FINAL_MODEL_PATH
)

print(
    "Selected Variant:",
    BEST_VARIANT,
    "—",
    BEST_VARIANT_NAME
)

print(
    "Selection metric:",
    "Highest Validation Macro F1"
)

## 18. Official Test Evaluation, Error Analysis and Export

Only the selected MLP is evaluated on the Official Test set.

Exports include:
- selected `.keras` model,
- StandardScaler `.joblib`,
- PCA `.joblib`,
- final result CSV,
- variant comparison CSV,
- classification report CSV,
- confusion matrix CSV,
- per-class metrics CSV,
- history CSV files,
- metadata JSON.

In [ ]:
best_model = (
    keras.models.load_model(
        FINAL_MODEL_PATH
    )
)

test_probs = (
    best_model.predict(
        X_test,
        verbose=0
    )
)

test_pred = np.argmax(
    test_probs,
    axis=1
)

TEST_METRICS = {
    "Test Accuracy": float(
        accuracy_score(
            y_test,
            test_pred
        )
    ),
    "Test Precision": float(
        precision_score(
            y_test,
            test_pred,
            average="macro",
            zero_division=0
        )
    ),
    "Test Recall": float(
        recall_score(
            y_test,
            test_pred,
            average="macro",
            zero_division=0
        )
    ),
    "Test F1": float(
        f1_score(
            y_test,
            test_pred,
            average="macro",
            zero_division=0
        )
    ),
}

display(
    pd.DataFrame([
        TEST_METRICS
    ])
)

report_dict = classification_report(
    y_test,
    test_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(
    report_dict
).transpose()

display(
    report_df
)

cm = confusion_matrix(
    y_test,
    test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"True {x}"
        for x in CLASS_NAMES
    ],
    columns=[
        f"Pred {x}"
        for x in CLASS_NAMES
    ]
)

display(
    cm_df
)

plt.figure(
    figsize=(8, 7)
)

plt.imshow(
    cm
)

plt.title(
    "Selected MLP — Official Test Confusion Matrix"
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    np.arange(
        NUM_CLASSES
    ),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(
        NUM_CLASSES
    ),
    CLASS_NAMES
)

for i in range(
    cm.shape[0]
):
    for j in range(
        cm.shape[1]
    ):
        plt.text(
            j,
            i,
            str(
                cm[i, j]
            ),
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()


per_class_rows = []

for class_name in CLASS_NAMES:
    values = report_dict[
        class_name
    ]

    per_class_rows.append({
        "Class": class_name,
        "Precision": float(
            values[
                "precision"
            ]
        ),
        "Recall": float(
            values[
                "recall"
            ]
        ),
        "F1": float(
            values[
                "f1-score"
            ]
        ),
        "Support": int(
            values[
                "support"
            ]
        ),
    })

per_class_df = pd.DataFrame(
    per_class_rows
)

display(
    per_class_df
)


member_result = pd.DataFrame([
    {
        "Member": OWNER_MEMBER_NO,
        "Student": MODEL_OWNER,
        "IT Number": OWNER_IT_NUMBER,
        "Model": "MLP",
        "Feature Pipeline": "HOG + GLCM + StandardScaler + PCA(95%)",
        "Selected Variant": BEST_VARIANT,
        "Selected Variant Name": BEST_VARIANT_NAME,
        "Validation F1": float(
            variant_results.loc[
                0,
                "Validation F1"
            ]
        ),
        **TEST_METRICS,
    }
])

display(
    member_result
)


FINAL_RESULT_CSV = (
    RESULTS_DIR
    / "it25100854_mlp_final_result.csv"
)

VARIANT_CSV = (
    RESULTS_DIR
    / "it25100854_mlp_variant_comparison.csv"
)

REPORT_CSV = (
    RESULTS_DIR
    / "it25100854_mlp_classification_report.csv"
)

CM_CSV = (
    RESULTS_DIR
    / "it25100854_mlp_confusion_matrix.csv"
)

PER_CLASS_CSV = (
    RESULTS_DIR
    / "it25100854_mlp_per_class_metrics.csv"
)

HISTORY_A_CSV = (
    RESULTS_DIR
    / "it25100854_mlp_history_A.csv"
)

HISTORY_B_CSV = (
    RESULTS_DIR
    / "it25100854_mlp_history_B.csv"
)

member_result.to_csv(
    FINAL_RESULT_CSV,
    index=False
)

variant_results.to_csv(
    VARIANT_CSV,
    index=False
)

report_df.to_csv(
    REPORT_CSV
)

cm_df.to_csv(
    CM_CSV
)

per_class_df.to_csv(
    PER_CLASS_CSV,
    index=False
)

history_a_df.to_csv(
    HISTORY_A_CSV,
    index=False
)

history_b_df.to_csv(
    HISTORY_B_CSV,
    index=False
)


metadata = {
    "model_owner": MODEL_OWNER,
    "owner_it_number": OWNER_IT_NUMBER,
    "owner_member_no": OWNER_MEMBER_NO,
    "member": OWNER_MEMBER_NO,
    "selected_model": "MLP",
    "selected_variant": BEST_VARIANT,
    "selected_variant_name": BEST_VARIANT_NAME,
    "selection_metric": "Validation Macro F1",
    "class_names": CLASS_NAMES,
    "dataset": KAGGLE_DATASET,
    "feature_pipeline": {
        "input": "Final group-preprocessed MRI",
        "grayscale_feature_size": [64, 64],
        "hog_orientations": HOG_ORIENTATIONS,
        "hog_pixels_per_cell": list(
            HOG_PIXELS_PER_CELL
        ),
        "hog_cells_per_block": list(
            HOG_CELLS_PER_BLOCK
        ),
        "glcm_distances": GLCM_DISTANCES,
        "glcm_angles_radians": GLCM_ANGLES,
        "glcm_properties": GLCM_PROPERTIES,
        "raw_feature_count": int(
            X_train_raw.shape[
                1
            ]
        ),
        "pca_components": int(
            X_train.shape[
                1
            ]
        ),
        "pca_variance_retained": float(
            pca.explained_variance_ratio_.sum()
        ),
    },
    "variant_configs": VARIANTS,
    "test_metrics": TEST_METRICS,
}

METADATA_PATH = (
    RESULTS_DIR
    / "best_model_metadata.json"
)

with open(
    METADATA_PATH,
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print(
    "Saved selected MLP:",
    FINAL_MODEL_PATH
)

print(
    "Saved scaler:",
    SCALER_PATH
)

print(
    "Saved PCA:",
    PCA_PATH
)

print(
    "Saved metadata:",
    METADATA_PATH
)

## 19. Download Final MLP Package + Viva Explanation

**Viva answer:**  
“We used HOG and GLCM to convert each MRI into numerical features, standardized those features using a scaler fitted only on Training data, and then applied PCA to retain 95% of the variance. We trained two MLP variants separately, compared them using Validation Macro F1, and used the Official Test set only after selecting the best variant.”

**Limitation:** engineered features may lose spatial information that CNNs can learn automatically.  
**Possible improvement:** tune hidden layers, dropout, PCA variance and regularization more systematically.

In [ ]:
from google.colab import files

PACKAGE_DIR = (
    WORK
    / "MLP_FINAL_PACKAGE"
)

if PACKAGE_DIR.exists():
    shutil.rmtree(
        PACKAGE_DIR
    )

PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

files_to_package = [
    FINAL_MODEL_PATH,
    SCALER_PATH,
    PCA_PATH,
    FINAL_RESULT_CSV,
    VARIANT_CSV,
    REPORT_CSV,
    CM_CSV,
    PER_CLASS_CSV,
    HISTORY_A_CSV,
    HISTORY_B_CSV,
    METADATA_PATH,
]

for src in files_to_package:
    shutil.copy2(
        src,
        PACKAGE_DIR
        / Path(
            src
        ).name
    )

PACKAGE_ZIP = Path(
    shutil.make_archive(
        "/content/MLP_FINAL_PACKAGE",
        "zip",
        root_dir=PACKAGE_DIR
    )
)

files.download(
    str(
        FINAL_MODEL_PATH
    )
)

files.download(
    str(
        FINAL_RESULT_CSV
    )
)

files.download(
    str(
        VARIANT_CSV
    )
)

files.download(
    str(
        METADATA_PATH
    )
)

files.download(
    str(
        PACKAGE_ZIP
    )
)